# African AI Risk Engine — JSE Single Market Prototype v1.0

**Date:** 2026-03-17
**Markets:** Johannesburg Stock Exchange (JSE) — Top 40 constituents
**Data Source:** Interactive Brokers TWS API (ib_insync, async/await)
**Methods:** Rolling window volatility, HAR-RV, Garman-Klass, Yang-Zhang, PCA, Gaussian HMM (pure NumPy), CSAD herding, multi-factor model, stress testing
**No GARCH. No EVT. No CSV/Excel loading.**


In [ ]:
import asyncio, nest_asyncio
nest_asyncio.apply()

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import FancyBboxPatch
from scipy import stats
from scipy.optimize import minimize
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import networkx as nx
import warnings, json, os
from datetime import datetime

OUTDIR = 'JSE_Risk_Output'
os.makedirs(OUTDIR, exist_ok=True)

WINDOWS = [21, 63, 126]
ANN = 252
CONF = [0.90, 0.95, 0.99]

JSE_TOP40 = [
    'AGL', 'AMS', 'ANG', 'ANH', 'BHP', 'BTI', 'CPI', 'DSY', 'EXX', 'FSR',
    'GFI', 'GLN', 'GRT', 'HAR', 'IMP', 'INL', 'INP', 'KIO', 'MNP', 'MRP',
    'MTN', 'NED', 'NPN', 'NRP', 'OMU', 'PRX', 'REM', 'SBK', 'SHP', 'SLM',
    'SOL', 'SSW', 'TBS', 'TFG', 'VOD', 'WHL', 'ABG', 'BID', 'CFR', 'MCG'
]

GLOBAL_SYMS = {
    'VIX': ('CBOE', 'IND', 'USD'),
    'DXY': ('NYBOT', 'IND', 'USD'),
}

FACTORS = {
    'F1_Global': 'PC1 of JSE stock returns (common shock)',
    'F2_USD': 'DXY + USDZAR (dollar pressure)',
    'F3_Commodity': 'Brent + Gold + Platinum + Copper',
    'F4_Sovereign': 'SA 10Y yield + USDZAR spread',
    'F5_Domestic': 'Banking index + sector rotation',
    'F6_Herding': 'CSAD cross-sectional absolute deviation'
}

SCORE_WEIGHTS = {
    'fwd_vol': 0.30,
    'pc1_conc': 0.20,
    'regime_stress': 0.20,
    'contagion': 0.15,
    'drawdown': 0.15
}

REGIME_NAMES = ['Stable Growth', 'Moderate', 'USD Tightening', 'Sovereign Stress', 'Systemic Crisis']
REGIME_COLORS = ['#2ecc71', '#3498db', '#f39c12', '#e67e22', '#e74c3c']

plt.rcParams.update({
    'figure.facecolor': '#0a0a0a',
    'axes.facecolor': '#0a0a0a',
    'axes.edgecolor': '#333',
    'axes.labelcolor': '#ccc',
    'text.color': '#ccc',
    'xtick.color': '#999',
    'ytick.color': '#999',
    'grid.color': '#222',
    'figure.dpi': 120,
    'font.size': 10
})


## Layer 1 — Data Aggregation (IBKR TWS API)

All market data is fetched **live** from Interactive Brokers TWS using `ib_insync` with native `async/await` (Python 3.14 compatible).

**Data fetched:**
- JSE Top 40 constituents (5Y daily OHLCV)
- JSE Top 40 Index (or ETF proxy)
- USDZAR exchange rate
- Commodity futures: Brent, Gold, Copper, Platinum
- Macro indices: DXY, VIX
- SA sovereign bond proxies

**Connection:** `127.0.0.1:7497` (TWS paper/live), clientId=2


In [ ]:
from ib_insync import *
import pandas as pd

# --- IBKR CONNECTION ---
ib = IB()
if not ib.isConnected():
    ib.connect('127.0.0.1', 7497, clientId=2)
print("Connected:", ib.isConnected())

# ==============================
# FETCH JSE TOP 40 STOCKS
# ==============================
all_stock_data = {}
failed_stocks = []

print(f"\nFetching {len(JSE_TOP40)} JSE stocks...", end=" ", flush=True)
fetched = 0

for sym in JSE_TOP40:
    try:
        contract = Stock(sym, 'JSE', 'ZAR')
        ib.qualifyContracts(contract)
        bars = ib.reqHistoricalData(
            contract, endDateTime='', durationStr='5 Y',
            barSizeSetting='1 day', whatToShow='TRADES',
            useRTH=True, formatDate=1
        )
        if bars:
            df = util.df(bars)
            df['date'] = pd.to_datetime(df['date'])
            all_stock_data[sym] = df.set_index('date')
            fetched += 1
        else:
            failed_stocks.append(sym)
    except:
        failed_stocks.append(sym)

print(f"done! {fetched}/{len(JSE_TOP40)} stocks fetched")

# ==============================
# FETCH JSE INDEX
# ==============================
print("Fetching JSE Index...", end=" ", flush=True)
jse_index = None

for idx in [Index('TOP40', 'JSE', 'ZAR'), Index('J200', 'JSE', 'ZAR'),
            Stock('STX40', 'JSE', 'ZAR'), Stock('STXSWX', 'JSE', 'ZAR')]:
    try:
        ib.qualifyContracts(idx)
        bars = ib.reqHistoricalData(
            idx, endDateTime='', durationStr='5 Y',
            barSizeSetting='1 day', whatToShow='TRADES',
            useRTH=True, formatDate=1
        )
        if bars:
            df = util.df(bars)
            df['date'] = pd.to_datetime(df['date'])
            jse_index = df.set_index('date')
            print(f"found ({idx.symbol})")
            break
    except:
        pass

if jse_index is None:
    closes = pd.DataFrame({s: d['close'] for s, d in all_stock_data.items()})
    proxy = closes.pct_change().mean(axis=1).add(1).cumprod() * 100
    jse_index = pd.DataFrame({'close': proxy})
    print("built equal-weight proxy")

# ==============================
# FETCH FX
# ==============================
print("Fetching USDZAR...", end=" ", flush=True)
fx_data = {}
try:
    fx = Forex('USDZAR')
    ib.qualifyContracts(fx)
    bars = ib.reqHistoricalData(
        fx, endDateTime='', durationStr='5 Y',
        barSizeSetting='1 day', whatToShow='MIDPOINT',
        useRTH=True, formatDate=1
    )
    if bars:
        df = util.df(bars)
        df['date'] = pd.to_datetime(df['date'])
        fx_data['USDZAR'] = df.set_index('date')['close']
        print("done")
except Exception as e:
    print(f"failed ({e})")

# ==============================
# FETCH COMMODITIES
# ==============================
print("Fetching commodities...", end=" ", flush=True)
commodity_data = {}
for name, contract in [
    ('Brent', ContFuture('COIL', 'IPE', currency='USD')),
    ('Gold', ContFuture('GC', 'COMEX', currency='USD')),
    ('Copper', ContFuture('HG', 'COMEX', currency='USD')),
    ('Platinum', ContFuture('PL', 'NYMEX', currency='USD')),
]:
    try:
        ib.qualifyContracts(contract)
        bars = ib.reqHistoricalData(
            contract, endDateTime='', durationStr='5 Y',
            barSizeSetting='1 day', whatToShow='TRADES',
            useRTH=True, formatDate=1
        )
        if bars:
            df = util.df(bars)
            df['date'] = pd.to_datetime(df['date'])
            commodity_data[name] = df.set_index('date')['close']
    except:
        pass
print(f"{len(commodity_data)}/4 fetched")

# ==============================
# FETCH MACRO (VIX, DXY)
# ==============================
print("Fetching macro...", end=" ", flush=True)
macro_data = {}
for name, contract in [('VIX', Index('VIX', 'CBOE', 'USD')),
                        ('DXY', Index('DX', 'NYBOT', 'USD'))]:
    try:
        ib.qualifyContracts(contract)
        bars = ib.reqHistoricalData(
            contract, endDateTime='', durationStr='5 Y',
            barSizeSetting='1 day', whatToShow='TRADES',
            useRTH=True, formatDate=1
        )
        if bars:
            df = util.df(bars)
            df['date'] = pd.to_datetime(df['date'])
            macro_data[name] = df.set_index('date')['close']
    except:
        pass
print(f"{len(macro_data)} fetched")

ib.disconnect()

# ==============================
# SUMMARY
# ==============================
print(f"\n{'='*50}")
print(f"  IBKR DATA SUMMARY")
print(f"{'='*50}")
print(f"  Stocks:      {len(all_stock_data)}/{len(JSE_TOP40)}  ({list(all_stock_data.keys())})")
print(f"  Failed:      {len(failed_stocks)}  ({failed_stocks[:5]}{'...' if len(failed_stocks)>5 else ''})")
print(f"  Index:       {'Yes' if jse_index is not None else 'No'}")
print(f"  FX:          {list(fx_data.keys())}")
print(f"  Commodities: {list(commodity_data.keys())}")
print(f"  Macro:       {list(macro_data.keys())}")
print(f"{'='*50}")

In [ ]:
closes = pd.DataFrame({sym: data['close'] for sym, data in all_stock_data.items()})
closes = closes.dropna(how='all').sort_index()

log_ret = np.log(closes / closes.shift(1)).dropna(how='all')

if 'close' in jse_index.columns:
    jse_ret = np.log(jse_index['close'] / jse_index['close'].shift(1)).dropna()
else:
    jse_ret = log_ret.mean(axis=1).dropna()

mkt_ret = log_ret.mean(axis=1).dropna()

print(f"{'='*60}")
print(f"  JSE MARKET DATA SUMMARY")
print(f"{'='*60}")
print(f"  Stocks available:    {len(all_stock_data)}")
print(f"  Date range:          {closes.index[0].date()} → {closes.index[-1].date()}")
print(f"  Trading days:        {len(closes)}")
print(f"  Missing stocks:      {failed_stocks}")
print(f"{'='*60}")

stats_df = pd.DataFrame({
    'Mean (ann%)': log_ret.mean() * ANN * 100,
    'Vol (ann%)': log_ret.std() * np.sqrt(ANN) * 100,
    'Skew': log_ret.skew(),
    'Kurt': log_ret.kurtosis(),
    'Sharpe': (log_ret.mean() / log_ret.std()) * np.sqrt(ANN)
}).round(2)
print("\nPer-Stock Summary:")
stats_df.sort_values('Vol (ann%)', ascending=False)


## Layer 0 — Portfolio Risk Core

### Rolling Window Volatility

$$\sigma_t = \sqrt{\frac{1}{w-1}\sum_{i=t-w+1}^{t}(R_i - \bar{R})^2}$$

$$\sigma_t^{(ann)} = \sigma_t \times \sqrt{N}$$

### HAR-RV (Heterogeneous Autoregressive Realized Variance)

$$RV_t^{(d)} = \alpha + \beta_d RV_{t-1}^{(d)} + \beta_w RV_{t-1}^{(w)} + \beta_m RV_{t-1}^{(m)} + \epsilon_t$$

### Value-at-Risk Suite

- **Historical VaR:** $VaR_\alpha = -\text{Percentile}(R, \alpha)$
- **CVaR (Expected Shortfall):** $CVaR_\alpha = -E[R \mid R \leq -VaR_\alpha]$
- **Cornish-Fisher:** $VaR_{CF} = \mu - \sigma\left(z_\alpha + \frac{z_\alpha^2-1}{6}S + \frac{z_\alpha^3-3z_\alpha}{24}K - \frac{2z_\alpha^3-5z_\alpha}{36}S^2\right)$

### OHLC Volatility Estimators

- **Garman-Klass:** $\sigma_{GK}^2 = \frac{1}{2}(\ln H/L)^2 - (2\ln 2-1)(\ln C/O)^2$
- **Yang-Zhang:** Combines open-to-close and close-to-open variance with Rogers-Satchell component.


In [ ]:
def rolling_vol(r, w=21):
    return r.rolling(w).std() * np.sqrt(ANN)

def garman_klass(df, w=21):
    log_hl = (np.log(df['high'] / df['low']))**2
    log_co = (np.log(df['close'] / df['open']))**2
    gk = 0.5 * log_hl - (2 * np.log(2) - 1) * log_co
    return np.sqrt(gk.rolling(w).mean() * ANN)

def yang_zhang(df, w=21):
    log_oc = np.log(df['open'] / df['close'].shift(1))
    log_co = np.log(df['close'] / df['open'])
    log_ho = np.log(df['high'] / df['open'])
    log_lo = np.log(df['low'] / df['open'])
    rs = log_ho * (log_ho - log_co) + log_lo * (log_lo - log_co)
    k = 0.34 / (1.34 + (w + 1) / (w - 1))
    oc_var = log_oc.rolling(w).var()
    co_var = log_co.rolling(w).var()
    rs_var = rs.rolling(w).mean()
    yz = oc_var + k * co_var + (1 - k) * rs_var
    return np.sqrt(yz.clip(lower=0) * ANN)

def har_rv(r, w_d=1, w_w=5, w_m=21, forecast_w=21):
    rv_d = r**2
    rv_w = rv_d.rolling(w_w).mean()
    rv_m = rv_d.rolling(w_m).mean()
    X = pd.DataFrame({'rv_d': rv_d.shift(1), 'rv_w': rv_w.shift(1), 'rv_m': rv_m.shift(1)}).dropna()
    y = rv_d.reindex(X.index)
    mask = X.notna().all(axis=1) & y.notna()
    X, y = X[mask], y[mask]
    if len(X) < 50:
        return pd.Series(dtype=float)
    from numpy.linalg import lstsq
    Xm = np.column_stack([np.ones(len(X)), X.values])
    beta, _, _, _ = lstsq(Xm, y.values, rcond=None)
    fitted = Xm @ beta
    return pd.Series(np.sqrt(np.maximum(fitted, 0)) * np.sqrt(ANN), index=X.index)

def hist_var_cvar(r, alpha=0.05):
    var = -np.percentile(r.dropna(), alpha * 100)
    cvar = -r[r <= -var].mean() if (r <= -var).any() else var
    return var, cvar

def param_var(r, alpha=0.05):
    z = stats.norm.ppf(alpha)
    return -(r.mean() + z * r.std())

def cf_var(r, alpha=0.05):
    z = stats.norm.ppf(alpha)
    s = r.skew()
    k = r.kurtosis()
    z_cf = z + (z**2 - 1)/6 * s + (z**3 - 3*z)/24 * k - (2*z**3 - 5*z)/36 * s**2
    return -(r.mean() + z_cf * r.std())

def rolling_dd(r):
    cum = (1 + r).cumprod()
    peak = cum.cummax()
    dd = (cum - peak) / peak
    return dd

def component_var(returns_df, alpha=0.05):
    w = np.ones(returns_df.shape[1]) / returns_df.shape[1]
    cov = returns_df.cov().values
    port_vol = np.sqrt(w @ cov @ w)
    z = stats.norm.ppf(alpha)
    port_var = -z * port_vol * np.sqrt(ANN)
    mcr = cov @ w / port_vol
    comp_var = w * mcr * (-z) * np.sqrt(ANN)
    hhi = np.sum((comp_var / comp_var.sum())**2)
    return pd.Series(comp_var, index=returns_df.columns), port_var, hhi


In [ ]:
print("=" * 60)
print("  LAYER 0 — PORTFOLIO RISK CORE")
print("=" * 60)

vol = {}
for w in WINDOWS:
    vol[w] = rolling_vol(mkt_ret, w)

har = har_rv(mkt_ret)

h_var, h_cvar = hist_var_cvar(mkt_ret)
p_var = param_var(mkt_ret)
c_var = cf_var(mkt_ret)

dd = rolling_dd(mkt_ret)

valid_ret = log_ret.dropna(axis=1, how='any').iloc[-252:]
if valid_ret.shape[1] >= 3:
    comp_var_s, port_var, hhi = component_var(valid_ret)
else:
    comp_var_s = pd.Series(dtype=float)
    port_var = h_var
    hhi = 1.0

gk_vols = {}
yz_vols = {}
for sym, data in all_stock_data.items():
    if all(c in data.columns for c in ['open', 'high', 'low', 'close']):
        gk_vols[sym] = garman_klass(data)
        yz_vols[sym] = yang_zhang(data)

print(f"\n  Rolling Vol (21d):  {vol[21].iloc[-1]:.2%}")
print(f"  Rolling Vol (63d):  {vol[63].iloc[-1]:.2%}")
print(f"  Rolling Vol (126d): {vol[126].iloc[-1]:.2%}")
print(f"  HAR-RV (latest):    {har.iloc[-1]:.2%}" if len(har) > 0 else "  HAR-RV: N/A")
print(f"\n  Historical VaR(95%): {h_var:.4f}")
print(f"  Historical CVaR:     {h_cvar:.4f}")
print(f"  Parametric VaR:      {p_var:.4f}")
print(f"  Cornish-Fisher VaR:  {c_var:.4f}")
print(f"\n  Max Drawdown:        {dd.min():.2%}")
print(f"  Current Drawdown:    {dd.iloc[-1]:.2%}")
print(f"  Herfindahl HHI:      {hhi:.4f}")
print(f"  Portfolio VaR (ann): {port_var:.4f}")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5), gridspec_kw={'width_ratios': [2, 1]})

ax = axes[0]
pos = mkt_ret.copy(); pos[pos < 0] = 0
neg = mkt_ret.copy(); neg[neg > 0] = 0
ax.fill_between(mkt_ret.index, pos, 0, color='#2ecc71', alpha=0.6, linewidth=0)
ax.fill_between(mkt_ret.index, neg, 0, color='#e74c3c', alpha=0.6, linewidth=0)
ax.set_title('JSE Market Daily Log Returns', fontsize=13, fontweight='bold')
ax.set_ylabel('Log Return')
ax.axhline(0, color='#555', linewidth=0.5)
ax.grid(True, alpha=0.15)

ax = axes[1]
ax.hist(mkt_ret.dropna(), bins=80, color='#3498db', alpha=0.7, density=True, edgecolor='none')
x_range = np.linspace(mkt_ret.min(), mkt_ret.max(), 200)
ax.plot(x_range, stats.norm.pdf(x_range, mkt_ret.mean(), mkt_ret.std()), color='#e74c3c', linewidth=2, label='Normal')
pct5 = np.percentile(mkt_ret.dropna(), 5)
ax.axvline(pct5, color='#f39c12', linestyle='--', linewidth=1.5, label=f'5th pct: {pct5:.4f}')
jb_stat, jb_p = stats.jarque_bera(mkt_ret.dropna())
ax.set_title(f'Distribution (JB={jb_stat:.0f}, p={jb_p:.2e})', fontsize=11)
ax.legend(fontsize=8)
ax.grid(True, alpha=0.15)

plt.tight_layout()
plt.savefig(os.path.join(OUTDIR, 'Fig01_Return_Distribution.png'), dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(16, 5))
colors_v = ['#2ecc71', '#3498db', '#e74c3c']
for i, w in enumerate(WINDOWS):
    ax.plot(vol[w].index, vol[w], color=colors_v[i], linewidth=1.2, label=f'{w}d ({vol[w].iloc[-1]:.1%})', alpha=0.85)
if len(har) > 0:
    ax.plot(har.index, har, color='#f39c12', linewidth=1, linestyle='--', label=f'HAR-RV ({har.iloc[-1]:.1%})', alpha=0.7)
ax.set_title('JSE Market Rolling Volatility (Annualized)', fontsize=13, fontweight='bold')
ax.set_ylabel('Volatility')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.15)
plt.tight_layout()
plt.savefig(os.path.join(OUTDIR, 'Fig02_Rolling_Volatility.png'), dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5), gridspec_kw={'width_ratios': [2, 1]})

ax = axes[0]
roll_w = 252
r_h_var = mkt_ret.rolling(roll_w).apply(lambda x: -np.percentile(x, 5), raw=True)
r_h_cvar = mkt_ret.rolling(roll_w).apply(lambda x: -x[x <= np.percentile(x, 5)].mean() if (x <= np.percentile(x, 5)).any() else -np.percentile(x, 5), raw=True)
r_p_var = mkt_ret.rolling(roll_w).apply(lambda x: -(np.mean(x) + stats.norm.ppf(0.05) * np.std(x)), raw=True)

ax.plot(r_h_var.index, r_h_var, color='#e74c3c', linewidth=1.2, label='Historical VaR(95%)')
ax.plot(r_h_cvar.index, r_h_cvar, color='#e67e22', linewidth=1.2, label='CVaR(95%)')
ax.plot(r_p_var.index, r_p_var, color='#3498db', linewidth=1.2, label='Parametric VaR')
ax.set_title('Rolling 252d VaR Suite', fontsize=13, fontweight='bold')
ax.set_ylabel('VaR (daily)')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.15)

ax = axes[1]
var_vals = [h_var, h_cvar, p_var, c_var]
var_labels = ['Hist VaR', 'CVaR', 'Param VaR', 'CF VaR']
colors_bar = ['#e74c3c', '#e67e22', '#3498db', '#9b59b6']
ax.barh(var_labels, var_vals, color=colors_bar, edgecolor='#333', height=0.6)
ax.set_title('Current VaR Comparison (95%)', fontsize=11)
ax.set_xlabel('Daily VaR')
for i, v in enumerate(var_vals):
    ax.text(v + 0.0001, i, f'{v:.4f}', va='center', fontsize=9, color='#ccc')
ax.grid(True, alpha=0.15, axis='x')

plt.tight_layout()
plt.savefig(os.path.join(OUTDIR, 'Fig03_VaR_Suite.png'), dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(16, 4))
ax.fill_between(dd.index, dd, 0, color='#e74c3c', alpha=0.5, linewidth=0)
ax.plot(dd.index, dd, color='#e74c3c', linewidth=0.5)
max_dd_idx = dd.idxmin()
ax.annotate(f'Max DD: {dd.min():.2%}\n{max_dd_idx.date()}',
            xy=(max_dd_idx, dd.min()), xytext=(max_dd_idx, dd.min() * 0.5),
            arrowprops=dict(arrowstyle='->', color='#f39c12'), fontsize=9, color='#f39c12')
roll_max_dd = dd.rolling(63).min()
ax.plot(roll_max_dd.index, roll_max_dd, color='#f39c12', linewidth=1, linestyle='--', label='63d Rolling Max DD', alpha=0.7)
ax.set_title('JSE Market Drawdown from Peak', fontsize=13, fontweight='bold')
ax.set_ylabel('Drawdown')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.15)
plt.tight_layout()
plt.savefig(os.path.join(OUTDIR, 'Fig04_Drawdown.png'), dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

ax = axes[0]
rep_sym = list(gk_vols.keys())[0] if gk_vols else None
if rep_sym:
    rv_s = rolling_vol(log_ret[rep_sym], 21)
    ax.plot(rv_s.index, rv_s, color='#3498db', linewidth=1, label='Rolling 21d', alpha=0.8)
    ax.plot(gk_vols[rep_sym].index, gk_vols[rep_sym], color='#2ecc71', linewidth=1, label='Garman-Klass', alpha=0.8)
    ax.plot(yz_vols[rep_sym].index, yz_vols[rep_sym], color='#e74c3c', linewidth=1, label='Yang-Zhang', alpha=0.8)
    ax.set_title(f'OHLC Volatility Estimators — {rep_sym}', fontsize=12, fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.15)
else:
    ax.text(0.5, 0.5, 'No OHLC data available', transform=ax.transAxes, ha='center', fontsize=12)

ax = axes[1]
if len(comp_var_s) > 0:
    top_cv = comp_var_s.abs().nlargest(15)
    colors_cv = ['#e74c3c' if v > 0 else '#2ecc71' for v in top_cv]
    ax.barh(top_cv.index, top_cv.values, color=colors_cv, edgecolor='#333', height=0.6)
    ax.set_title(f'Component VaR (top 15) — HHI={hhi:.3f}', fontsize=11, fontweight='bold')
    ax.set_xlabel('Component VaR')
    ax.grid(True, alpha=0.15, axis='x')
else:
    ax.text(0.5, 0.5, 'Insufficient stocks for Component VaR', transform=ax.transAxes, ha='center', fontsize=12)

plt.tight_layout()
plt.savefig(os.path.join(OUTDIR, 'Fig05_OHLC_CompVaR.png'), dpi=150, bbox_inches='tight')
plt.show()


## Layer 2 — Structural Correlation & PCA

### PCA Eigenvalue Decomposition

Given the return matrix $\mathbf{R} \in \mathbb{R}^{T \times N}$, the covariance matrix $\Sigma = \frac{1}{T-1}\mathbf{R}^\top \mathbf{R}$ is decomposed as:

$$\Sigma = V \Lambda V^\top$$

where $\Lambda = \text{diag}(\lambda_1, \lambda_2, \ldots, \lambda_N)$ are eigenvalues and $V$ the eigenvectors (principal components).

### PC1 Concentration Ratio

$$CR_1 = \frac{\lambda_1}{\sum_{i=1}^{N} \lambda_i}$$

**Fragility threshold:** $CR_1 > 0.60$ signals systemic fragility — a single factor dominates all stock co-movement.

### Correlation Network

Eigenvector centrality identifies **contagion hubs** — stocks whose failure propagates most widely.


In [ ]:
print("=" * 60)
print("  LAYER 2 — STRUCTURAL CORRELATION & PCA")
print("=" * 60)

valid_stocks = log_ret.dropna(axis=1, thresh=int(len(log_ret)*0.7))
ret_clean = valid_stocks.dropna()

scaler = StandardScaler()
ret_scaled = scaler.fit_transform(ret_clean)

pca = PCA()
pca.fit(ret_scaled)
loadings = pd.DataFrame(pca.components_.T, index=ret_clean.columns,
                         columns=[f'PC{i+1}' for i in range(len(ret_clean.columns))])
explained = pca.explained_variance_ratio_

print(f"  PC1 explains: {explained[0]:.1%}")
print(f"  PC1+PC2:      {explained[:2].sum():.1%}")
print(f"  PC1+PC2+PC3:  {explained[:3].sum():.1%}")

def rolling_pc1(ret_df, w=252):
    conc = pd.Series(index=ret_df.index[w:], dtype=float)
    for i in range(w, len(ret_df)):
        chunk = ret_df.iloc[i-w:i].dropna(axis=1)
        if chunk.shape[1] < 3:
            continue
        s = StandardScaler().fit_transform(chunk)
        p = PCA(n_components=min(5, chunk.shape[1])).fit(s)
        conc.iloc[i-w] = p.explained_variance_ratio_[0]
    return conc

pc1_conc = rolling_pc1(valid_stocks)
fragility = (pc1_conc > 0.60).sum() / len(pc1_conc) * 100 if len(pc1_conc) > 0 else 0

corr_mat = ret_clean.iloc[-252:].corr()

G = nx.Graph()
for i in range(len(corr_mat)):
    for j in range(i+1, len(corr_mat)):
        w_val = abs(corr_mat.iloc[i, j])
        if w_val > 0.3:
            G.add_edge(corr_mat.index[i], corr_mat.columns[j], weight=w_val)

if len(G.nodes) > 0:
    eigen_cent = nx.eigenvector_centrality(G, weight='weight', max_iter=1000)
    between_cent = nx.betweenness_centrality(G, weight='weight')
    hub = max(eigen_cent, key=eigen_cent.get)
    print(f"\n  Contagion Hub: {hub} (eigenvector centrality: {eigen_cent[hub]:.3f})")
    print(f"  Network edges: {G.number_of_edges()}")
    print(f"  Fragility periods (PC1>60%): {fragility:.1f}%")
else:
    eigen_cent = {}
    between_cent = {}
    print("  Network: insufficient correlations")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

ax = axes[0]
n_show = min(15, len(explained))
ax.bar(range(n_show), explained[:n_show] * 100, color='#3498db', edgecolor='#333', alpha=0.8)
ax2 = ax.twinx()
ax2.plot(range(n_show), np.cumsum(explained[:n_show]) * 100, color='#e74c3c', marker='o', markersize=4, linewidth=2)
ax2.set_ylabel('Cumulative %', color='#e74c3c')
ax2.tick_params(axis='y', colors='#e74c3c')
ax.set_title('PCA Scree Plot', fontsize=13, fontweight='bold')
ax.set_xlabel('Principal Component')
ax.set_ylabel('Variance Explained (%)')
ax.grid(True, alpha=0.15)

ax = axes[1]
n_pcs = min(5, loadings.shape[1])
n_stocks = min(20, loadings.shape[0])
top_load = loadings.iloc[:n_stocks, :n_pcs]
im = ax.imshow(top_load.values, cmap='RdBu_r', aspect='auto', vmin=-0.5, vmax=0.5)
ax.set_xticks(range(n_pcs))
ax.set_xticklabels([f'PC{i+1}' for i in range(n_pcs)])
ax.set_yticks(range(n_stocks))
ax.set_yticklabels(top_load.index, fontsize=7)
ax.set_title('PC Loadings (top stocks)', fontsize=12, fontweight='bold')
plt.colorbar(im, ax=ax, shrink=0.8)

plt.tight_layout()
plt.savefig(os.path.join(OUTDIR, 'Fig06_PCA_Scree.png'), dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
fig = plt.figure(figsize=(18, 6))
gs = gridspec.GridSpec(1, 3, width_ratios=[1, 1, 1.2])

ax = fig.add_subplot(gs[0])
n_corr = min(20, len(corr_mat))
cm = corr_mat.iloc[:n_corr, :n_corr]
im = ax.imshow(cm.values, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
ax.set_xticks(range(n_corr))
ax.set_xticklabels(cm.index, rotation=90, fontsize=6)
ax.set_yticks(range(n_corr))
ax.set_yticklabels(cm.index, fontsize=6)
ax.set_title('Correlation Matrix (252d)', fontsize=11, fontweight='bold')
plt.colorbar(im, ax=ax, shrink=0.7)

ax = fig.add_subplot(gs[1])
if len(pc1_conc) > 0:
    ax.plot(pc1_conc.index, pc1_conc, color='#3498db', linewidth=1.2)
    ax.axhline(0.60, color='#e74c3c', linestyle='--', linewidth=1, label='Fragility (0.60)')
    ax.fill_between(pc1_conc.index, pc1_conc, 0.60,
                     where=pc1_conc > 0.60, color='#e74c3c', alpha=0.3)
    ax.set_title('Rolling PC1 Concentration', fontsize=11, fontweight='bold')
    ax.set_ylabel('PC1 Variance Ratio')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.15)

ax = fig.add_subplot(gs[2])
if len(G.nodes) > 0:
    pos_g = nx.spring_layout(G, k=2, seed=42, weight='weight')
    node_sizes = [eigen_cent.get(n, 0.01) * 2000 + 100 for n in G.nodes()]
    node_colors = [eigen_cent.get(n, 0) for n in G.nodes()]
    nx.draw_networkx_nodes(G, pos_g, ax=ax, node_size=node_sizes, node_color=node_colors,
                           cmap=plt.cm.YlOrRd, alpha=0.8, edgecolors='#555')
    nx.draw_networkx_labels(G, pos_g, ax=ax, font_size=6, font_color='#ccc')
    edges = G.edges(data=True)
    edge_w = [d['weight'] * 2 for _, _, d in edges]
    nx.draw_networkx_edges(G, pos_g, ax=ax, width=edge_w, alpha=0.3, edge_color='#999')
    ax.set_title(f'Correlation Network (hub: {hub})', fontsize=11, fontweight='bold')
else:
    ax.text(0.5, 0.5, 'Insufficient data', transform=ax.transAxes, ha='center')
ax.set_facecolor('#0a0a0a')

plt.tight_layout()
plt.savefig(os.path.join(OUTDIR, 'Fig07_Correlation_Network.png'), dpi=150, bbox_inches='tight')
plt.show()


## Layer 3 — African Factor Model (6 Factors)

### Factor Regression

$$R_i = \alpha + \beta_1 F_1 + \beta_2 F_2 + \cdots + \beta_k F_k + \epsilon$$

### Risk Contribution per Factor

$$RC_j = \beta_j \times \text{Cov}(F_j, R_i)$$

### Factor Weight

$$Weight_j = \frac{\beta_j \times \text{Cov}(F_j, R_i)}{\text{Var}(R_i)}$$

### CSAD Herding Detection

$$CSAD_t = \alpha + \gamma_1 |R_{m,t}| + \gamma_2 R_{m,t}^2 + \epsilon_t$$

Herding measure: $H = -\gamma_2$. A significant negative $\gamma_2$ implies **herding behavior** — investors crowd into the same trades during stress.

| Factor | Description |
|--------|------------|
| F1_Global | PC1 of JSE stock returns (common shock) |
| F2_USD | DXY + USDZAR (dollar pressure) |
| F3_Commodity | Brent + Gold + Platinum + Copper |
| F4_Sovereign | SA 10Y yield + USDZAR spread |
| F5_Domestic | Banking index vs mining (sector rotation) |
| F6_Herding | CSAD cross-sectional absolute deviation |


In [ ]:
print("=" * 60)
print("  LAYER 3 — AFRICAN FACTOR MODEL (6 FACTORS)")
print("=" * 60)

common_idx = mkt_ret.index

# F1: Global Risk Factor (PC1 projection)
if len(ret_clean) > 0:
    pc1_scores = ret_scaled @ pca.components_[0]
    f1 = pd.Series(pc1_scores, index=ret_clean.index, name='F1_Global')
else:
    f1 = pd.Series(0, index=common_idx, name='F1_Global')

# F2: USD Liquidity Factor
f2_components = []
if 'USDZAR' in fx_data:
    f2_components.append(fx_data['USDZAR'].pct_change())
if 'DXY' in macro_data:
    f2_components.append(macro_data['DXY'].pct_change())
if f2_components:
    f2_df = pd.concat(f2_components, axis=1).dropna()
    f2 = f2_df.mean(axis=1).rename('F2_USD')
else:
    f2 = pd.Series(0, index=common_idx, name='F2_USD')

# F3: Commodity Factor
f3_components = []
for comm in ['Brent', 'Gold', 'Copper', 'Platinum']:
    if comm in commodity_data:
        f3_components.append(commodity_data[comm].pct_change())
if f3_components:
    f3_df = pd.concat(f3_components, axis=1).dropna()
    f3 = f3_df.mean(axis=1).rename('F3_Commodity')
else:
    f3 = pd.Series(0, index=common_idx, name='F3_Commodity')

# F4: Sovereign Factor
if 'USDZAR' in fx_data:
    zar = fx_data['USDZAR']
    f4 = zar.pct_change().rolling(21).std().rename('F4_Sovereign')
else:
    f4 = pd.Series(0, index=common_idx, name='F4_Sovereign')

# F5: Domestic Factor (banking vs mining)
banking = [s for s in ['FSR', 'SBK', 'NED', 'ABG', 'CPI'] if s in all_stock_data]
mining = [s for s in ['AGL', 'AMS', 'ANG', 'GFI', 'IMP', 'HAR'] if s in all_stock_data]
if banking and mining:
    bank_ret = log_ret[banking].mean(axis=1)
    mine_ret = log_ret[mining].mean(axis=1)
    f5 = (bank_ret - mine_ret).rename('F5_Domestic')
else:
    f5 = pd.Series(0, index=common_idx, name='F5_Domestic')

# F6: Herding Factor (CSAD)
print("\n  Computing CSAD Herding (all available JSE stocks)...")

def compute_csad(stock_returns, market_return, window=60):
    csad = stock_returns.sub(market_return, axis=0).abs().mean(axis=1)
    abs_rm = market_return.abs()
    rm_sq = market_return ** 2
    gamma2_roll = pd.Series(index=csad.index, dtype=float)
    pval_roll = pd.Series(index=csad.index, dtype=float)
    for i in range(window, len(csad)):
        idx = csad.index[i-window:i]
        y = csad.loc[idx].values
        x1 = abs_rm.reindex(idx).values
        x2 = rm_sq.reindex(idx).values
        mask = np.isfinite(y) & np.isfinite(x1) & np.isfinite(x2)
        if mask.sum() < 20:
            continue
        X = np.column_stack([np.ones(mask.sum()), x1[mask], x2[mask]])
        y_clean = y[mask]
        try:
            beta = np.linalg.lstsq(X, y_clean, rcond=None)[0]
            resid = y_clean - X @ beta
            se = np.sqrt(np.diag(np.linalg.inv(X.T @ X) * np.var(resid)))
            gamma2_roll.iloc[i] = beta[2]
            pval_roll.iloc[i] = 2 * (1 - stats.t.cdf(abs(beta[2] / se[2]), df=mask.sum()-3))
        except:
            continue
    herding = -gamma2_roll
    return csad, herding, gamma2_roll, pval_roll

csad_ret = log_ret.dropna(axis=1, thresh=int(len(log_ret)*0.5))
csad_val, herding, gamma2, pvals = compute_csad(csad_ret, mkt_ret)
f6 = herding.rename('F6_Herding')

latest_herding = herding.dropna().iloc[-1] if len(herding.dropna()) > 0 else 0
herding_level = 'Strong' if latest_herding > 0.05 else ('Moderate' if latest_herding > 0.02 else 'None')
print(f"  CSAD stocks used: {csad_ret.shape[1]}")
print(f"  Latest herding index: {latest_herding:.4f} ({herding_level})")

factors = pd.concat([f1, f2, f3, f4, f5, f6], axis=1).dropna(how='all')
factor_names = ['F1_Global', 'F2_USD', 'F3_Commodity', 'F4_Sovereign', 'F5_Domestic', 'F6_Herding']

factor_z = (factors - factors.rolling(63).mean()) / factors.rolling(63).std()

print("\n  Factor Regression (OLS):")
common = pd.concat([mkt_ret.rename('Rm'), factors], axis=1).dropna()
if len(common) > 100:
    y = common['Rm'].values
    X = np.column_stack([np.ones(len(common)), common[factor_names].values])
    betas = np.linalg.lstsq(X, y, rcond=None)[0]
    y_hat = X @ betas
    r2 = 1 - np.var(y - y_hat) / np.var(y)
    factor_betas = pd.Series(betas[1:], index=factor_names)
    for i, fn in enumerate(factor_names):
        cov_fj_ri = np.cov(common[fn].values, y)[0, 1]
        rc = factor_betas[fn] * cov_fj_ri
        weight = rc / np.var(y)
        print(f"    {fn:20s}  β={factor_betas[fn]:+.4f}  RC={rc:.6f}  Weight={weight:.2%}")
    print(f"\n    R² = {r2:.4f}")
else:
    factor_betas = pd.Series(0, index=factor_names)
    r2 = 0
    print("    Insufficient data for regression")

def classify_shock(fz_row):
    labels = []
    if abs(fz_row.get('F3_Commodity', 0)) > 1.5:
        labels.append('SUPPLY')
    if abs(fz_row.get('F2_USD', 0)) > 1.5 or abs(fz_row.get('F4_Sovereign', 0)) > 1.5:
        labels.append('FINANCIAL')
    if abs(fz_row.get('F1_Global', 0)) > 1.5:
        labels.append('DEMAND')
    return ' / '.join(labels) if labels else 'NONE'

latest_fz = factor_z.iloc[-1] if len(factor_z) > 0 else pd.Series(0, index=factor_names)
shock_type = classify_shock(latest_fz)
print(f"\n  Current Shock Classification: {shock_type}")


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

ax = axes[0]
if hasattr(factor_betas, 'index') and len(factor_betas) > 0:
    colors_fb = ['#e74c3c' if b < 0 else '#2ecc71' for b in factor_betas]
    ax.barh(factor_betas.index, factor_betas.values, color=colors_fb, edgecolor='#333', height=0.5)
    ax.set_title('Factor Betas', fontsize=12, fontweight='bold')
    ax.axvline(0, color='#555', linewidth=0.5)
    ax.grid(True, alpha=0.15, axis='x')

ax = axes[1]
if len(latest_fz) > 0:
    fz_vals = [latest_fz.get(fn, 0) for fn in factor_names]
    colors_fz = ['#e74c3c' if z > 1.5 else '#f39c12' if z > 0.5 else '#2ecc71' if z > -0.5 else '#3498db' if z > -1.5 else '#9b59b6' for z in fz_vals]
    ax.barh(factor_names, fz_vals, color=colors_fz, edgecolor='#333', height=0.5)
    ax.axvline(0, color='#555', linewidth=0.5)
    ax.axvline(1.5, color='#e74c3c', linestyle='--', linewidth=0.8, alpha=0.5)
    ax.axvline(-1.5, color='#e74c3c', linestyle='--', linewidth=0.8, alpha=0.5)
    ax.set_title(f'Current Factor Z-Scores [{shock_type}]', fontsize=11, fontweight='bold')
    ax.grid(True, alpha=0.15, axis='x')

ax = axes[2]
if len(common) > 100:
    rc_vals = []
    for fn in factor_names:
        cov_fj = np.cov(common[fn].values, common['Rm'].values)[0, 1]
        rc_vals.append(abs(factor_betas[fn] * cov_fj))
    rc_arr = np.array(rc_vals)
    rc_pct = rc_arr / rc_arr.sum() * 100 if rc_arr.sum() > 0 else rc_arr
    colors_pie = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12', '#9b59b6', '#e67e22']
    wedges, texts, autotexts = ax.pie(rc_pct, labels=factor_names, autopct='%1.1f%%',
                                       colors=colors_pie, textprops={'fontsize': 7, 'color': '#ccc'})
    ax.set_title('Factor Risk Contribution', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(OUTDIR, 'Fig08_Factor_Betas.png'), dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(16, 8), gridspec_kw={'height_ratios': [2, 1]})

ax = axes[0]
fz_plot = factor_z.dropna(how='all')
if len(fz_plot) > 0:
    colors_ft = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12', '#9b59b6', '#e67e22']
    for i, fn in enumerate(factor_names):
        if fn in fz_plot.columns:
            roll = fz_plot[fn].rolling(21).mean()
            ax.plot(roll.index, roll, color=colors_ft[i], linewidth=1, label=fn, alpha=0.8)
    ax.axhline(1.5, color='#e74c3c', linestyle='--', linewidth=0.8, alpha=0.5)
    ax.axhline(-1.5, color='#e74c3c', linestyle='--', linewidth=0.8, alpha=0.5)
    ax.axhline(0, color='#555', linewidth=0.5)
    ax.set_title('Rolling 21d Factor Z-Scores', fontsize=13, fontweight='bold')
    ax.legend(fontsize=8, ncol=3)
    ax.grid(True, alpha=0.15)

ax = axes[1]
shock_ts = fz_plot.apply(lambda row: classify_shock(row), axis=1)
shock_map = {'NONE': 0, 'SUPPLY': 1, 'FINANCIAL': 2, 'DEMAND': 3}
for combo in shock_ts.unique():
    mask = shock_ts == combo
    if combo == 'NONE':
        continue
    color = '#e74c3c' if 'FINANCIAL' in combo else '#f39c12' if 'SUPPLY' in combo else '#9b59b6'
    ax.scatter(shock_ts.index[mask], [1]*mask.sum(), color=color, s=3, alpha=0.6, label=combo)
ax.set_title('Shock Classification Timeline', fontsize=11, fontweight='bold')
ax.set_yticks([])
ax.legend(fontsize=7, ncol=4, loc='upper left')
ax.grid(True, alpha=0.15)

plt.tight_layout()
plt.savefig(os.path.join(OUTDIR, 'Fig09_Factor_Timeline.png'), dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(16, 9), sharex=True)

ax = axes[0]
if len(csad_val.dropna()) > 0:
    ax.plot(csad_val.index, csad_val, color='#3498db', linewidth=0.8, alpha=0.8)
    ax.set_title('CSAD (Cross-Sectional Absolute Deviation)', fontsize=12, fontweight='bold')
    ax.set_ylabel('CSAD')
    ax.grid(True, alpha=0.15)

ax = axes[1]
herd_clean = herding.dropna()
if len(herd_clean) > 0:
    ax.plot(herd_clean.index, herd_clean, color='#e74c3c', linewidth=0.8)
    ax.axhline(0.05, color='#e74c3c', linestyle='--', linewidth=1, label='Strong (0.05)')
    ax.axhline(0.02, color='#f39c12', linestyle='--', linewidth=1, label='Moderate (0.02)')
    ax.axhline(0, color='#555', linewidth=0.5)
    ax.fill_between(herd_clean.index, herd_clean, 0.05,
                     where=herd_clean > 0.05, color='#e74c3c', alpha=0.3)
    ax.set_title(f'Herding Index (current: {latest_herding:.4f} — {herding_level})', fontsize=12, fontweight='bold')
    ax.set_ylabel('Herding (-γ₂)')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.15)

ax = axes[2]
pval_clean = pvals.dropna()
if len(pval_clean) > 0:
    sig = pval_clean < 0.05
    ax.scatter(pval_clean.index[sig], pval_clean[sig], color='#e74c3c', s=3, alpha=0.6, label='p<0.05')
    ax.scatter(pval_clean.index[~sig], pval_clean[~sig], color='#555', s=1, alpha=0.3, label='p≥0.05')
    ax.axhline(0.05, color='#f39c12', linestyle='--', linewidth=1)
    ax.set_title('γ₂ Significance (p-values)', fontsize=11, fontweight='bold')
    ax.set_ylabel('p-value')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.15)

plt.tight_layout()
plt.savefig(os.path.join(OUTDIR, 'Fig10_CSAD_Herding.png'), dpi=150, bbox_inches='tight')
plt.show()


## Layer 4 — HMM Regime Detection (5-State Gaussian)

A 5-state **Gaussian Hidden Markov Model** is fit via Baum-Welch (EM) on a feature matrix of 19 market features. Viterbi decoding extracts the most likely regime path.

**States:**
1. Stable Growth — low vol, positive drift
2. Moderate — normal market conditions
3. USD Tightening — dollar strength, EM pressure
4. Sovereign Stress — SA-specific fiscal/political risk
5. Systemic Crisis — high vol, contagion, drawdown

**Algorithm:**
- Forward-backward (log-space) for posterior state probabilities
- Viterbi for MAP path
- Multiple random seeds, best log-likelihood selected
- States reordered by ascending 21d volatility


In [ ]:
class GaussianHMM_NP:
    def __init__(self, K=5, max_iter=100, tol=1e-4, seed=42):
        self.K = K
        self.max_iter = max_iter
        self.tol = tol
        self.rng = np.random.RandomState(seed)

    def _init_params(self, X):
        N, D = X.shape
        self.pi = np.ones(self.K) / self.K
        self.A = np.full((self.K, self.K), 1/self.K) * 0.1 + np.eye(self.K) * 0.9
        idx = self.rng.choice(N, self.K, replace=False)
        self.mu = X[idx].copy()
        self.var = np.tile(X.var(axis=0), (self.K, 1)) + 1e-6

    def _log_B(self, X):
        N, D = X.shape
        log_b = np.zeros((N, self.K))
        for k in range(self.K):
            diff = X - self.mu[k]
            log_b[:, k] = -0.5 * (D * np.log(2*np.pi) +
                           np.sum(np.log(self.var[k])) +
                           np.sum(diff**2 / self.var[k], axis=1))
        return log_b

    def _forward(self, log_b):
        N = log_b.shape[0]
        log_alpha = np.full((N, self.K), -np.inf)
        log_alpha[0] = np.log(self.pi + 1e-300) + log_b[0]
        for t in range(1, N):
            for k in range(self.K):
                log_alpha[t, k] = np.logaddexp.reduce(
                    log_alpha[t-1] + np.log(self.A[:, k] + 1e-300)
                ) + log_b[t, k]
        return log_alpha

    def _backward(self, log_b):
        N = log_b.shape[0]
        log_beta = np.full((N, self.K), -np.inf)
        log_beta[-1] = 0
        for t in range(N-2, -1, -1):
            for k in range(self.K):
                log_beta[t, k] = np.logaddexp.reduce(
                    np.log(self.A[k] + 1e-300) + log_b[t+1] + log_beta[t+1]
                )
        return log_beta

    def _estep(self, X):
        log_b = self._log_B(X)
        log_alpha = self._forward(log_b)
        log_beta = self._backward(log_b)
        log_gamma = log_alpha + log_beta
        log_gamma -= np.logaddexp.reduce(log_gamma, axis=1, keepdims=True)
        gamma = np.exp(log_gamma)
        ll = np.logaddexp.reduce(log_alpha[-1])
        return gamma, log_alpha, log_beta, log_b, ll

    def _mstep(self, X, gamma):
        Nk = gamma.sum(axis=0) + 1e-300
        self.pi = gamma[0] / gamma[0].sum()
        self.mu = (gamma.T @ X) / Nk[:, None]
        for k in range(self.K):
            diff = X - self.mu[k]
            self.var[k] = (gamma[:, k:k+1] * diff**2).sum(axis=0) / Nk[k] + 1e-6
        N = len(X)
        log_b = self._log_B(X)
        log_alpha = self._forward(log_b)
        log_beta = self._backward(log_b)
        for i in range(self.K):
            for j in range(self.K):
                log_xi_sum = np.logaddexp.reduce([
                    log_alpha[t, i] + np.log(self.A[i, j] + 1e-300) +
                    log_b[t+1, j] + log_beta[t+1, j]
                    for t in range(N-1)
                ])
                self.A[i, j] = np.exp(log_xi_sum)
        self.A /= self.A.sum(axis=1, keepdims=True) + 1e-300

    def fit(self, X):
        self._init_params(X)
        prev_ll = -np.inf
        for iteration in range(self.max_iter):
            gamma, _, _, _, ll = self._estep(X)
            self._mstep(X, gamma)
            if abs(ll - prev_ll) < self.tol:
                break
            prev_ll = ll
        self.ll_ = ll
        return self

    def predict(self, X):
        log_b = self._log_B(X)
        N = log_b.shape[0]
        V = np.full((N, self.K), -np.inf)
        ptr = np.zeros((N, self.K), dtype=int)
        V[0] = np.log(self.pi + 1e-300) + log_b[0]
        for t in range(1, N):
            for k in range(self.K):
                trans = V[t-1] + np.log(self.A[:, k] + 1e-300)
                ptr[t, k] = np.argmax(trans)
                V[t, k] = trans[ptr[t, k]] + log_b[t, k]
        path = np.zeros(N, dtype=int)
        path[-1] = np.argmax(V[-1])
        for t in range(N-2, -1, -1):
            path[t] = ptr[t+1, path[t+1]]
        return path

    def predict_proba(self, X):
        gamma, _, _, _, _ = self._estep(X)
        return gamma


In [ ]:
# DEBUG: check index alignment before HMM
import warnings
warnings.filterwarnings('ignore')

samples = {
    'mkt_ret': mkt_ret.index,
    'log_ret': log_ret.index,
}
if len(factor_z) > 0:
    samples['factor_z'] = factor_z.index
if len(pc1_conc) > 0:
    samples['pc1_conc'] = pc1_conc.index

for name, idx in samples.items():
    print(f"{name:12s} | type={idx.dtype} | tz={getattr(idx, 'tz', 'N/A')} | len={len(idx)} | first={idx[0]} | last={idx[-1]}")

# Test: what happens with a simple 2-way concat?
v1 = rolling_vol(mkt_ret, 21).dropna()
if len(factor_z) > 0 and 'F1_Global' in factor_z.columns:
    f1_col = factor_z['F1_Global'].dropna()
    test = pd.concat([v1, f1_col], axis=1).dropna()
    print(f"\nTest concat vol_21d + F1_Global: {len(test)} rows")
    print(f"  vol_21d index dtype: {v1.index.dtype}, sample: {v1.index[0]}")
    print(f"  F1_Global index dtype: {f1_col.index.dtype}, sample: {f1_col.index[0]}")
    # Check string representation
    v1_dates = set(str(d)[:10] for d in v1.index)
    f1_dates = set(str(d)[:10] for d in f1_col.index)
    print(f"  Overlapping date strings: {len(v1_dates & f1_dates)}")
    print(f"  v1 unique dates: {len(v1_dates)}, f1 unique dates: {len(f1_dates)}")

In [ ]:
print("=" * 60)
print("  LAYER 4 — HMM REGIME DETECTION")
print("=" * 60)

features = []
feat_names_hmm = []

for w in WINDOWS:
    v = rolling_vol(mkt_ret, w)
    features.append(v)
    feat_names_hmm.append(f'vol_{w}d')

top_stocks = list(all_stock_data.keys())[:5]
for sym in top_stocks:
    if sym in log_ret.columns:
        v = rolling_vol(log_ret[sym], 21)
        features.append(v)
        feat_names_hmm.append(f'vol_{sym}')

for fn in factor_names:
    if fn in factor_z.columns:
        features.append(factor_z[fn])
        feat_names_hmm.append(fn)

if len(pc1_conc) > 0:
    features.append(pc1_conc)
    feat_names_hmm.append('PC1_conc')

# Build common date grid from the densest source (market returns vol)
common_idx = rolling_vol(mkt_ret, max(WINDOWS)).dropna().index

# Reindex all features to the common grid, forward-filling sparse series
for i in range(len(features)):
    s = features[i].copy()
    # Strip timezone if present
    if hasattr(s.index, 'tz') and s.index.tz is not None:
        s.index = s.index.tz_localize(None)
    if hasattr(s.index, 'normalize'):
        s.index = s.index.normalize()
    s = s[~s.index.duplicated(keep='first')]
    # Reindex to common grid with forward fill (last known value)
    s = s.reindex(common_idx, method='ffill')
    features[i] = s

feat_df = pd.concat(features, axis=1)
feat_df.columns = feat_names_hmm
feat_df = feat_df.dropna()

print(f"  Feature matrix: {feat_df.shape[0]} obs × {feat_df.shape[1]} features")
print(f"  Features: {feat_names_hmm}")

if feat_df.shape[0] == 0:
    raise ValueError("Feature matrix is empty — check factor_z and pc1_conc data upstream")

scaler_hmm = StandardScaler()
X_hmm = scaler_hmm.fit_transform(feat_df.values)

best_ll = -np.inf
best_model = None
K = 5

for seed in range(5):
    try:
        model = GaussianHMM_NP(K=K, max_iter=200, tol=1e-5, seed=seed*42+1)
        model.fit(X_hmm)
        if model.ll_ > best_ll:
            best_ll = model.ll_
            best_model = model
    except:
        continue

states = best_model.predict(X_hmm)
probs = best_model.predict_proba(X_hmm)

vol_feat_idx = feat_names_hmm.index('vol_21d')
state_vol = [X_hmm[states == k, vol_feat_idx].mean() if (states == k).any() else 0 for k in range(K)]
order = np.argsort(state_vol)
remap = {old: new for new, old in enumerate(order)}
states = np.array([remap[s] for s in states])
probs = probs[:, order]
best_model.A = best_model.A[order][:, order]
best_model.mu = best_model.mu[order]
best_model.var = best_model.var[order]

regime_s = pd.Series(states, index=feat_df.index, name='regime')
reg_probs = pd.DataFrame(probs, index=feat_df.index, columns=REGIME_NAMES)
trans_m = best_model.A

current_regime = REGIME_NAMES[states[-1]]
crisis_prob = probs[-1, 3] + probs[-1, 4]

p_now = probs[-1]
p1 = p_now @ trans_m
p5 = p_now @ np.linalg.matrix_power(trans_m, 5)
p21 = p_now @ np.linalg.matrix_power(trans_m, 21)

print(f"\n  Current Regime: {current_regime}")
print(f"  Crisis Probability: {crisis_prob:.2%}")
print(f"  Log-likelihood: {best_ll:.2f}")
print(f"\n  Regime Forecast:")
print(f"    +1d:  {REGIME_NAMES[np.argmax(p1)]} (p={p1.max():.2%})")
print(f"    +5d:  {REGIME_NAMES[np.argmax(p5)]} (p={p5.max():.2%})")
print(f"    +21d: {REGIME_NAMES[np.argmax(p21)]} (p={p21.max():.2%})")

print(f"\n  Transition Matrix:")
for i, name in enumerate(REGIME_NAMES):
    row = ' '.join([f'{trans_m[i,j]:.3f}' for j in range(K)])
    print(f"    {name:20s} → [{row}]")

In [ ]:
fig, ax = plt.subplots(figsize=(16, 5))
aligned_ret = mkt_ret.reindex(regime_s.index)
for i, name in enumerate(REGIME_NAMES):
    mask = regime_s == i
    if mask.any():
        ax.fill_between(aligned_ret.index, aligned_ret.where(mask, 0), 0,
                         color=REGIME_COLORS[i], alpha=0.6, label=name, linewidth=0)
ax.set_title('JSE Market Returns — Regime Colored', fontsize=13, fontweight='bold')
ax.set_ylabel('Log Return')
ax.legend(fontsize=8, ncol=5, loc='upper left')
ax.grid(True, alpha=0.15)
plt.tight_layout()
plt.savefig(os.path.join(OUTDIR, 'Fig11_Regime_Returns.png'), dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
fig = plt.figure(figsize=(18, 10))
gs = gridspec.GridSpec(2, 3, hspace=0.35, wspace=0.3)

# Stacked regime probabilities
ax = fig.add_subplot(gs[0, :2])
for i, name in enumerate(REGIME_NAMES):
    bottom = reg_probs.iloc[:, :i].sum(axis=1) if i > 0 else 0
    ax.fill_between(reg_probs.index, bottom, bottom + reg_probs[name],
                     color=REGIME_COLORS[i], alpha=0.7, label=name)
ax.set_title('Regime Probabilities (stacked)', fontsize=12, fontweight='bold')
ax.set_ylabel('Probability')
ax.legend(fontsize=7, ncol=5, loc='upper left')
ax.grid(True, alpha=0.15)

# Current regime pie
ax = fig.add_subplot(gs[0, 2])
ax.pie(probs[-1], labels=REGIME_NAMES, colors=REGIME_COLORS, autopct='%1.0f%%',
       textprops={'fontsize': 7, 'color': '#ccc'})
ax.set_title(f'Current: {current_regime}', fontsize=11, fontweight='bold')

# Transition heatmap
ax = fig.add_subplot(gs[1, 0])
im = ax.imshow(trans_m, cmap='YlOrRd', vmin=0, vmax=1, aspect='auto')
for i in range(K):
    for j in range(K):
        ax.text(j, i, f'{trans_m[i,j]:.2f}', ha='center', va='center', fontsize=7,
                color='white' if trans_m[i,j] > 0.5 else '#ccc')
ax.set_xticks(range(K))
ax.set_xticklabels([n[:8] for n in REGIME_NAMES], fontsize=7, rotation=45)
ax.set_yticks(range(K))
ax.set_yticklabels([n[:8] for n in REGIME_NAMES], fontsize=7)
ax.set_title('Transition Matrix', fontsize=11, fontweight='bold')

# Regime forecast bars
ax = fig.add_subplot(gs[1, 1])
x_pos = np.arange(K)
width = 0.25
ax.bar(x_pos - width, p1, width, color=REGIME_COLORS, alpha=0.9, label='+1d')
ax.bar(x_pos, p5, width, color=REGIME_COLORS, alpha=0.6, label='+5d')
ax.bar(x_pos + width, p21, width, color=REGIME_COLORS, alpha=0.3, label='+21d')
ax.set_xticks(x_pos)
ax.set_xticklabels([n[:8] for n in REGIME_NAMES], fontsize=7, rotation=45)
ax.set_title('Regime Forecast', fontsize=11, fontweight='bold')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.15, axis='y')

# Per-regime statistics
ax = fig.add_subplot(gs[1, 2])
ax.axis('off')
aligned_ret_r = mkt_ret.reindex(regime_s.index)
table_data = []
for i, name in enumerate(REGIME_NAMES):
    mask = regime_s == i
    r = aligned_ret_r[mask]
    if len(r) > 0:
        table_data.append([name[:12], f'{len(r)}', f'{r.mean()*252*100:.1f}%',
                          f'{r.std()*np.sqrt(252)*100:.1f}%', f'{r.min()*100:.2f}%'])
    else:
        table_data.append([name[:12], '0', 'N/A', 'N/A', 'N/A'])
tbl = ax.table(cellText=table_data, colLabels=['Regime', 'Days', 'Ann Ret', 'Ann Vol', 'Worst Day'],
               loc='center', cellLoc='center')
tbl.auto_set_font_size(False)
tbl.set_fontsize(8)
for key, cell in tbl.get_celld().items():
    cell.set_facecolor('#1a1a1a')
    cell.set_edgecolor('#333')
    cell.set_text_props(color='#ccc')
ax.set_title('Per-Regime Statistics', fontsize=11, fontweight='bold')

plt.savefig(os.path.join(OUTDIR, 'Fig12_Regime_Dashboard.png'), dpi=150, bbox_inches='tight')
plt.show()


## Layer 5 — Adaptive Risk Score (0–100)

### Sigmoid Transformation

$$Score = \frac{100}{1 + e^{-0.5 \cdot z_{composite}}}$$

where $z_{composite}$ is the weighted sum of standardized risk components:

| Component | Weight | Description |
|-----------|--------|-------------|
| fwd_vol | 30% | Forward-looking 21d rolling volatility |
| pc1_conc | 20% | PC1 concentration ratio (systemic fragility) |
| regime_stress | 20% | P(Sovereign Stress) + P(Systemic Crisis) |
| contagion | 15% | Mean pairwise rolling correlation |
| drawdown | 15% | Current drawdown from peak |

**Risk Bands:** 0–40 LOW, 40–60 MODERATE, 60–75 HIGH, 75–100 CRITICAL


In [ ]:
print("=" * 60)
print("  LAYER 5 — ADAPTIVE RISK SCORE (0-100)")
print("=" * 60)

common_idx_rs = feat_df.index

fwd_vol_raw = vol[21].reindex(common_idx_rs)
pc1_raw = pc1_conc.reindex(common_idx_rs)
regime_stress_raw = (reg_probs['Sovereign Stress'] + reg_probs['Systemic Crisis']).reindex(common_idx_rs)

if log_ret.shape[1] >= 2:
    roll_corr = log_ret.rolling(60).corr()
    contagion_raw = pd.Series(index=common_idx_rs, dtype=float)
    for date in common_idx_rs:
        if date in roll_corr.index.get_level_values(0):
            c = roll_corr.loc[date]
            mask_c = np.triu(np.ones(c.shape), k=1).astype(bool)
            contagion_raw[date] = c.values[mask_c].mean() if mask_c.sum() > 0 else 0
else:
    contagion_raw = pd.Series(0, index=common_idx_rs)

dd_raw = dd.reindex(common_idx_rs).abs()

def z_score(s):
    return (s - s.expanding().mean()) / s.expanding().std()

components = {
    'fwd_vol': z_score(fwd_vol_raw),
    'pc1_conc': z_score(pc1_raw),
    'regime_stress': z_score(regime_stress_raw),
    'contagion': z_score(contagion_raw),
    'drawdown': z_score(dd_raw)
}

raw_score = sum(SCORE_WEIGHTS[k] * components[k].fillna(0) for k in SCORE_WEIGHTS)
risk_score = 100 / (1 + np.exp(-0.5 * raw_score))

current_score = risk_score.iloc[-1]
if current_score < 40:
    band = 'LOW RISK'
elif current_score < 60:
    band = 'MODERATE'
elif current_score < 75:
    band = 'HIGH RISK'
else:
    band = 'CRITICAL'

print(f"\n  Current Risk Score: {current_score:.1f} / 100")
print(f"  Risk Band: {band}")
print(f"\n  Component Contributions:")
for k, w in SCORE_WEIGHTS.items():
    val = components[k].iloc[-1] if len(components[k].dropna()) > 0 else 0
    print(f"    {k:20s}  weight={w:.0%}  z={val:+.2f}  contribution={w*val:+.3f}")


In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(16, 11), gridspec_kw={'height_ratios': [2, 1.2, 0.8]})

ax = axes[0]
ax.plot(risk_score.index, risk_score, color='#ccc', linewidth=0.8)
ax.fill_between(risk_score.index, risk_score, 50,
                 where=risk_score >= 75, color='#e74c3c', alpha=0.5, label='Critical (75+)')
ax.fill_between(risk_score.index, risk_score, 50,
                 where=(risk_score >= 60) & (risk_score < 75), color='#e67e22', alpha=0.5, label='High (60-75)')
ax.fill_between(risk_score.index, risk_score, 50,
                 where=(risk_score >= 40) & (risk_score < 60), color='#f39c12', alpha=0.4, label='Moderate (40-60)')
ax.fill_between(risk_score.index, risk_score, 50,
                 where=risk_score < 40, color='#2ecc71', alpha=0.4, label='Low (<40)')
ax.axhline(75, color='#e74c3c', linestyle='--', linewidth=0.8, alpha=0.5)
ax.axhline(60, color='#e67e22', linestyle='--', linewidth=0.8, alpha=0.5)
ax.axhline(40, color='#f39c12', linestyle='--', linewidth=0.8, alpha=0.5)
ax.set_title(f'African AI Risk Score — Current: {current_score:.1f} [{band}]', fontsize=14, fontweight='bold')
ax.set_ylabel('Risk Score (0-100)')
ax.set_ylim(0, 100)
ax.legend(fontsize=8, ncol=4)
ax.grid(True, alpha=0.15)

ax = axes[1]
comp_colors = {'fwd_vol': '#e74c3c', 'pc1_conc': '#3498db', 'regime_stress': '#f39c12',
               'contagion': '#9b59b6', 'drawdown': '#e67e22'}
bottom = pd.Series(0, index=common_idx_rs)
for k in SCORE_WEIGHTS:
    contrib = (SCORE_WEIGHTS[k] * components[k].fillna(0)).clip(lower=0)
    ax.fill_between(contrib.index, bottom, bottom + contrib, color=comp_colors[k], alpha=0.6, label=k)
    bottom = bottom + contrib
ax.set_title('Component Contributions (positive only)', fontsize=11, fontweight='bold')
ax.legend(fontsize=7, ncol=5)
ax.grid(True, alpha=0.15)

ax = axes[2]
ax.hist(risk_score.dropna(), bins=60, color='#3498db', alpha=0.7, edgecolor='none', density=True)
ax.axvline(current_score, color='#e74c3c', linewidth=2, label=f'Current: {current_score:.1f}')
ax.set_title('Risk Score Distribution', fontsize=11, fontweight='bold')
ax.set_xlabel('Score')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.15)

plt.tight_layout()
plt.savefig(os.path.join(OUTDIR, 'Fig13_Risk_Score.png'), dpi=150, bbox_inches='tight')
plt.show()


## Layer 6 — Africa-Specific Stress Testing

| # | Scenario | Shocked Factors | Rationale |
|---|----------|-----------------|-----------|
| 1 | Oil Collapse -40% | F3 -3.0σ, F4 +2.0σ | Commodity-dependent fiscal revenue collapse |
| 2 | USD +5% Shock | F2 +3.0σ, F4 +1.5σ | EM capital flight, ZAR depreciation |
| 3 | US Rate Hike +100bps | F2 +2.0σ, F1 -1.5σ | Global risk-off, dollar strength |
| 4 | Sovereign Yield Spike | F4 +3.0σ, F2 +1.0σ | SA fiscal deterioration |
| 5 | China Slowdown | F3 -2.5σ, F1 -2.0σ | Commodity demand collapse |
| 6 | Gold Rally +30% | F3 +2.5σ | Safe-haven flow, mining sector boost |
| 7 | ZAR Flash Crash -15% | F2 +4.0σ, F4 +3.0σ, F5 -2.0σ | Currency crisis |
| 8 | Mining Sector Collapse | F3 -3.0σ, F5 -2.5σ | Sector-specific shock |
| 9 | EM Contagion | F1 -3.0σ, F2 +2.5σ, F4 +2.0σ | Emerging market sell-off |
| 10 | Load Shedding Crisis | F5 -3.0σ, F4 +1.5σ | SA-specific infrastructure failure |

Propagation uses 3-round contagion with 20% decay through the correlation matrix.


In [ ]:
print("=" * 60)
print("  LAYER 6 — AFRICA-SPECIFIC STRESS TESTING")
print("=" * 60)

SCENARIOS = [
    ('Oil Collapse -40%',        {'F3_Commodity': -3.0, 'F4_Sovereign': 2.0}),
    ('USD +5% Shock',            {'F2_USD': 3.0, 'F4_Sovereign': 1.5}),
    ('US Rate Hike +100bps',     {'F2_USD': 2.0, 'F1_Global': -1.5}),
    ('Sovereign Yield Spike',    {'F4_Sovereign': 3.0, 'F2_USD': 1.0}),
    ('China Slowdown',           {'F3_Commodity': -2.5, 'F1_Global': -2.0}),
    ('Gold Rally +30%',          {'F3_Commodity': 2.5}),
    ('ZAR Flash Crash -15%',     {'F2_USD': 4.0, 'F4_Sovereign': 3.0, 'F5_Domestic': -2.0}),
    ('Mining Sector Collapse',   {'F3_Commodity': -3.0, 'F5_Domestic': -2.5}),
    ('EM Contagion',             {'F1_Global': -3.0, 'F2_USD': 2.5, 'F4_Sovereign': 2.0}),
    ('Load Shedding Crisis',     {'F5_Domestic': -3.0, 'F4_Sovereign': 1.5}),
]

def run_stress(scenarios, betas, corr_matrix, n_rounds=3, decay=0.20):
    results = {}
    for name, shocks in scenarios:
        impact = 0
        for f, shock_z in shocks.items():
            if f in betas.index:
                impact += betas[f] * shock_z * mkt_ret.std()
        total = impact
        for rnd in range(n_rounds):
            total += decay * total * corr_matrix.mean().mean() if len(corr_matrix) > 0 else 0
        results[name] = total * 100
    return pd.Series(results)

stress_results = run_stress(SCENARIOS, factor_betas, corr_mat)

print(f"\n  {'Scenario':<30s}  Impact (%)")
print(f"  {'-'*45}")
for name, impact in stress_results.sort_values().items():
    bar = '█' * int(abs(impact) * 2)
    sign = '↓' if impact < 0 else '↑'
    print(f"  {name:<30s}  {impact:+6.2f}% {sign} {bar}")


In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
sorted_stress = stress_results.sort_values()
colors_st = ['#e74c3c' if v < -1 else '#e67e22' if v < 0 else '#2ecc71' for v in sorted_stress]
ax.barh(sorted_stress.index, sorted_stress.values, color=colors_st, edgecolor='#333', height=0.6)
ax.axvline(0, color='#555', linewidth=0.5)
for i, (name, val) in enumerate(sorted_stress.items()):
    ax.text(val + (0.05 if val >= 0 else -0.05), i, f'{val:+.2f}%',
            va='center', ha='left' if val >= 0 else 'right', fontsize=9, color='#ccc')
ax.set_title('Africa-Specific Stress Test — Portfolio Impact', fontsize=13, fontweight='bold')
ax.set_xlabel('Impact (%)')
ax.grid(True, alpha=0.15, axis='x')
plt.tight_layout()
plt.savefig(os.path.join(OUTDIR, 'Fig14_Stress_Test.png'), dpi=150, bbox_inches='tight')
plt.show()


## Layer 7 — AI Portfolio Intelligence Report

Comprehensive JSON payload and formatted text report summarizing all risk layers. This output is designed for API consumption by portfolio management systems.


In [ ]:
print("=" * 70)
print("  LAYER 7 — AI PORTFOLIO INTELLIGENCE REPORT")
print("  African AI Risk Engine — JSE Single Market Prototype")
print("=" * 70)

report = {
    'metadata': {
        'engine': 'African AI Risk Engine v1.0',
        'market': 'JSE (Johannesburg Stock Exchange)',
        'date': str(pd.Timestamp.now().date()),
        'stocks_available': len(all_stock_data),
        'stocks_failed': failed_stocks,
        'data_range': f"{closes.index[0].date()} → {closes.index[-1].date()}"
    },
    'risk_score': {
        'score': round(current_score, 1),
        'band': band,
        'components': {k: round(float(components[k].iloc[-1]), 4) for k in SCORE_WEIGHTS if len(components[k].dropna()) > 0}
    },
    'regime': {
        'current': current_regime,
        'crisis_probability': round(float(crisis_prob), 4),
        'forecast_1d': REGIME_NAMES[np.argmax(p1)],
        'forecast_5d': REGIME_NAMES[np.argmax(p5)],
        'forecast_21d': REGIME_NAMES[np.argmax(p21)]
    },
    'volatility': {
        'vol_21d': round(float(vol[21].iloc[-1]), 4),
        'vol_63d': round(float(vol[63].iloc[-1]), 4),
        'vol_126d': round(float(vol[126].iloc[-1]), 4),
        'var_95': round(float(h_var), 6),
        'cvar_95': round(float(h_cvar), 6),
        'max_drawdown': round(float(dd.min()), 4),
        'current_drawdown': round(float(dd.iloc[-1]), 4)
    },
    'factors': {
        'betas': {fn: round(float(factor_betas.get(fn, 0)), 4) for fn in factor_names},
        'current_z': {fn: round(float(latest_fz.get(fn, 0)), 2) for fn in factor_names},
        'r_squared': round(float(r2), 4),
        'shock_classification': shock_type
    },
    'herding': {
        'index': round(float(latest_herding), 4),
        'level': herding_level,
        'stocks_used': int(csad_ret.shape[1])
    },
    'pca': {
        'pc1_explained': round(float(explained[0]), 4),
        'pc1_concentration': round(float(pc1_conc.iloc[-1]), 4) if len(pc1_conc) > 0 else None,
        'fragility_flag': bool(pc1_conc.iloc[-1] > 0.60) if len(pc1_conc) > 0 else False
    },
    'stress_test': {
        'worst_scenario': stress_results.idxmin(),
        'worst_impact': round(float(stress_results.min()), 2),
        'all_scenarios': {k: round(float(v), 2) for k, v in stress_results.items()}
    },
    'recommendation': ''
}

if current_score >= 75:
    rec = "CRITICAL — Maximum defensive allocation. Reduce equity to minimum. Increase liquidity buffer to 25%+. Hedge ZAR exposure."
elif current_score >= 60:
    rec = "HIGH RISK — Reduce equity exposure by 20-30%. Increase cash allocation. Monitor regime transitions daily."
elif current_score >= 40:
    rec = "MODERATE — Maintain current allocation with elevated monitoring. Consider tactical hedges on concentrated positions."
else:
    rec = "LOW RISK — Standard allocation appropriate. Rebalance to target weights. Opportunistic entry points possible."

report['recommendation'] = rec

with open(os.path.join(OUTDIR, 'api_payload.json'), 'w') as f:
    json.dump(report, f, indent=2, default=str)

_box = []
_box.append("")
_box.append("  +---------------------------------------------------+")
_box.append(f"  |  RISK SCORE:  {current_score:5.1f} / 100  [{band:^15s}]  |")
_box.append(f"  |  REGIME:      {current_regime:<35s} |")
_box.append(f"  |  CRISIS P:    {crisis_prob:5.2%}{'':30s} |")
_box.append("  +---------------------------------------------------+")
_box.append("")
_box.append("  VOLATILITY")
_box.append(f"    21d: {vol[21].iloc[-1]:.2%}  |  63d: {vol[63].iloc[-1]:.2%}  |  126d: {vol[126].iloc[-1]:.2%}")
_box.append(f"    VaR(95%): {h_var:.4f}  |  CVaR: {h_cvar:.4f}  |  Max DD: {dd.min():.2%}")
_box.append("")
_box.append("  FACTOR Z-SCORES (current)")
print("\n".join(_box))
for fn in factor_names:
    z = latest_fz.get(fn, 0)
    bar_len = int(abs(z) * 5)
    bar_ch = chr(9608) * bar_len
    direction = chr(8594) if abs(z) < 0.5 else (chr(8593) if z > 0 else chr(8595))
    print(f"    {fn:20s}  z={z:+5.2f} {direction} {bar_ch}")

_tail = []
_tail.append("")
_tail.append(f"  HERDING:  {herding_level} (index={latest_herding:.4f}, {csad_ret.shape[1]} stocks)")
_tail.append(f"  SHOCK:    {shock_type}")
_tail.append("")
_tail.append("  STRESS TEST")
_tail.append(f"    Worst: {stress_results.idxmin()} ({stress_results.min():+.2f}%)")
_tail.append("")
_tail.append("  RECOMMENDATION")
_tail.append(f"    {rec}")
_tail.append("")
_tail.append(f"  -> Full payload saved to {OUTDIR}/api_payload.json")
_tail.append("")
print("\n".join(_tail))


In [ ]:
fig = plt.figure(figsize=(24, 18))
gs = gridspec.GridSpec(4, 3, hspace=0.4, wspace=0.3)

# Panel 1: JSE returns with regime coloring
ax = fig.add_subplot(gs[0, 0])
ar = mkt_ret.reindex(regime_s.index)
for i, name in enumerate(REGIME_NAMES):
    mask = regime_s == i
    if mask.any():
        ax.fill_between(ar.index, ar.where(mask, 0), 0, color=REGIME_COLORS[i], alpha=0.6, linewidth=0)
ax.set_title('Returns (regime-colored)', fontsize=10, fontweight='bold')
ax.grid(True, alpha=0.15)
ax.tick_params(labelsize=7)

# Panel 2: Rolling volatility
ax = fig.add_subplot(gs[0, 1])
for i, w in enumerate(WINDOWS):
    ax.plot(vol[w].index, vol[w], color=['#2ecc71','#3498db','#e74c3c'][i], linewidth=0.8, label=f'{w}d')
ax.set_title('Rolling Volatility', fontsize=10, fontweight='bold')
ax.legend(fontsize=7)
ax.grid(True, alpha=0.15)
ax.tick_params(labelsize=7)

# Panel 3: Risk score timeline
ax = fig.add_subplot(gs[0, 2])
ax.plot(risk_score.index, risk_score, color='#ccc', linewidth=0.8)
ax.fill_between(risk_score.index, risk_score, 50, where=risk_score>=75, color='#e74c3c', alpha=0.5)
ax.fill_between(risk_score.index, risk_score, 50, where=(risk_score>=60)&(risk_score<75), color='#e67e22', alpha=0.5)
ax.fill_between(risk_score.index, risk_score, 50, where=(risk_score>=40)&(risk_score<60), color='#f39c12', alpha=0.4)
ax.fill_between(risk_score.index, risk_score, 50, where=risk_score<40, color='#2ecc71', alpha=0.4)
ax.set_ylim(0, 100)
ax.set_title(f'Risk Score: {current_score:.1f} [{band}]', fontsize=10, fontweight='bold')
ax.grid(True, alpha=0.15)
ax.tick_params(labelsize=7)

# Panel 4: Regime probabilities stacked
ax = fig.add_subplot(gs[1, 0])
for i, name in enumerate(REGIME_NAMES):
    bottom = reg_probs.iloc[:, :i].sum(axis=1) if i > 0 else 0
    ax.fill_between(reg_probs.index, bottom, bottom + reg_probs[name], color=REGIME_COLORS[i], alpha=0.7)
ax.set_title('Regime Probabilities', fontsize=10, fontweight='bold')
ax.grid(True, alpha=0.15)
ax.tick_params(labelsize=7)

# Panel 5: Factor z-scores bar
ax = fig.add_subplot(gs[1, 1])
fz_vals = [latest_fz.get(fn, 0) for fn in factor_names]
colors_fzb = ['#e74c3c' if abs(z) > 1.5 else '#f39c12' if abs(z) > 0.5 else '#2ecc71' for z in fz_vals]
ax.barh(factor_names, fz_vals, color=colors_fzb, edgecolor='#333', height=0.5)
ax.axvline(0, color='#555', linewidth=0.5)
ax.set_title('Factor Z-Scores', fontsize=10, fontweight='bold')
ax.grid(True, alpha=0.15, axis='x')
ax.tick_params(labelsize=7)

# Panel 6: Correlation heatmap
ax = fig.add_subplot(gs[1, 2])
nc = min(15, len(corr_mat))
cm_sub = corr_mat.iloc[:nc, :nc]
im = ax.imshow(cm_sub.values, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
ax.set_xticks(range(nc))
ax.set_xticklabels(cm_sub.index, rotation=90, fontsize=5)
ax.set_yticks(range(nc))
ax.set_yticklabels(cm_sub.index, fontsize=5)
ax.set_title('Correlation (252d)', fontsize=10, fontweight='bold')

# Panel 7: Component VaR
ax = fig.add_subplot(gs[2, 0])
if len(comp_var_s) > 0:
    top_cv2 = comp_var_s.abs().nlargest(10)
    ax.barh(top_cv2.index, top_cv2.values, color='#e74c3c', edgecolor='#333', height=0.5)
    ax.set_title(f'Component VaR (HHI={hhi:.3f})', fontsize=10, fontweight='bold')
    ax.grid(True, alpha=0.15, axis='x')
else:
    ax.text(0.5, 0.5, 'N/A', transform=ax.transAxes, ha='center')
ax.tick_params(labelsize=7)

# Panel 8: Transition matrix heatmap
ax = fig.add_subplot(gs[2, 1])
im2 = ax.imshow(trans_m, cmap='YlOrRd', vmin=0, vmax=1, aspect='auto')
for i in range(K):
    for j in range(K):
        ax.text(j, i, f'{trans_m[i,j]:.2f}', ha='center', va='center', fontsize=6,
                color='white' if trans_m[i,j] > 0.5 else '#ccc')
ax.set_xticks(range(K))
ax.set_xticklabels([n[:7] for n in REGIME_NAMES], fontsize=6, rotation=45)
ax.set_yticks(range(K))
ax.set_yticklabels([n[:7] for n in REGIME_NAMES], fontsize=6)
ax.set_title('Transition Matrix', fontsize=10, fontweight='bold')

# Panel 9: Stress test bar chart
ax = fig.add_subplot(gs[2, 2])
ss = stress_results.sort_values()
colors_ss = ['#e74c3c' if v < -1 else '#e67e22' if v < 0 else '#2ecc71' for v in ss]
ax.barh(ss.index, ss.values, color=colors_ss, edgecolor='#333', height=0.5)
ax.axvline(0, color='#555', linewidth=0.5)
ax.set_title('Stress Test Impact', fontsize=10, fontweight='bold')
ax.grid(True, alpha=0.15, axis='x')
ax.tick_params(labelsize=6)

# Panel 10: CSAD herding
ax = fig.add_subplot(gs[3, 0])
hc = herding.dropna()
if len(hc) > 0:
    ax.plot(hc.index, hc, color='#e74c3c', linewidth=0.8)
    ax.axhline(0.05, color='#e74c3c', linestyle='--', linewidth=0.8, alpha=0.5)
    ax.axhline(0.02, color='#f39c12', linestyle='--', linewidth=0.8, alpha=0.5)
ax.set_title(f'Herding: {herding_level}', fontsize=10, fontweight='bold')
ax.grid(True, alpha=0.15)
ax.tick_params(labelsize=7)

# Panel 11: Drawdown
ax = fig.add_subplot(gs[3, 1])
ax.fill_between(dd.index, dd, 0, color='#e74c3c', alpha=0.5, linewidth=0)
ax.set_title(f'Drawdown (max: {dd.min():.2%})', fontsize=10, fontweight='bold')
ax.grid(True, alpha=0.15)
ax.tick_params(labelsize=7)

# Panel 12: Text summary
ax = fig.add_subplot(gs[3, 2])
ax.axis('off')
_sep = chr(9472) * 35
_lines = [
    "AFRICAN AI RISK ENGINE",
    "JSE Single Market Prototype v1.0",
    _sep,
    f"Risk Score:     {current_score:.1f} / 100 [{band}]",
    f"Regime:         {current_regime}",
    f"Crisis P:       {crisis_prob:.2%}",
    f"Vol (21d):      {vol[21].iloc[-1]:.2%}",
    f"VaR (95%):      {h_var:.4f}",
    f"Max Drawdown:   {dd.min():.2%}",
    f"Herding:        {herding_level}",
    f"Shock:          {shock_type}",
    f"Worst Stress:   {stress_results.idxmin()}",
    f"                ({stress_results.min():+.2f}%)",
    _sep,
    f"{rec[:60]}",
    f"{rec[60:] if len(rec) > 60 else ''}",
]
summary_text = "\n".join(_lines)
ax.text(0.05, 0.95, summary_text, transform=ax.transAxes, fontsize=8,
        verticalalignment='top', fontfamily='monospace', color='#2ecc71',
        bbox=dict(boxstyle='round', facecolor='#111', edgecolor='#2ecc71', alpha=0.9))

plt.savefig(os.path.join(OUTDIR, 'Fig15_Master_Dashboard.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f"\n  ✓ Master Dashboard saved to {OUTDIR}/Fig15_Master_Dashboard.png")
